In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohammad2012191/q2-ka-ai-2026")

print("Path to dataset files:", path)

In [ ]:
import os
import pandas as pd
import numpy as np
from PIL import Image
from sklearn.model_selection import train_test_split
import kagglehub
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load Labels
labels_df = pd.read_csv(os.path.join(path, "labels.csv"))
img_dir = os.path.join(path, "images")

images = []
ages = []

# Load Images and Resize
print("Loading images...")
for index, row in labels_df.iterrows():
    img_name = row.iloc[0]
    age = row.iloc[1]
    img_path = os.path.join(img_dir, img_name)

    if os.path.exists(img_path):
        img = Image.open(img_path).convert('RGB')
        img_array = np.array(img) / 255.0 # Normalize pixel values to [0, 1]
        images.append(img_array)
        ages.append(age)

X = np.array(images)
y = np.array(ages)

# Transpose image dimensions to match PyTorch format (N, C, H, W)
X = np.transpose(X, (0, 3, 1, 2))

# Split Data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# 1. Convert Numpy arrays to PyTorch Tensors
X = np.array(images, dtype=np.float32)
y = np.array(ages, dtype=np.float32)

# Train / Test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(X_train.shape, X_test.shape)

In [ ]:
# 2. Create TensorDataset objects
train_ds = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
test_ds   = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))



In [ ]:
# 3. Create DataLoaders

train_loader = DataLoader(train_ds, batch_size=32, shuffle=True)
test_loader   = DataLoader(test_ds, batch_size=32)

In [ ]:
# 4. Print shape of one batch

# Print dataset sizes
print("Train dataset:", len(train_ds))
print("Test dataset:", len(test_ds))

# Get the first batch from the training DataLoader
X_batch, y_batch = next(iter(train_loader))
print(f"Training batch input shape: {X_batch.shape}")
print(f"Training batch labels shape: {y_batch.shape}")

In [ ]:
# 5. Display sample images

# Display the first 6 images in the batch
plt.figure(figsize=(8, 4))

for i in range(6):
    plt.subplot(2, 3, i + 1)

    # Convert from (C, H, W) to (H, W, C) for matplotlib
    img = images[i].permute(1, 2, 0)

    plt.imshow(img)
    plt.title(f"Label: {labels[i].item()}")
    plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
# Task 1: Write your model class here:
class AgeRegressor(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Linear(256, 64),
            nn.ReLU(),
            nn.Linear(64, 1)
        )

    def forward(self, x):
        return self.model(x).squeeze(1)

In [ ]:
# Task 2: Write your training loop here:
def train_loop(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        preds = model(x)
        loss = criterion(preds, y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
# Task 3: Write your validation loop here:
def val_loop(model, loader, criterion, device):
    model.eval()
    total_loss = 0
    with torch.no_grad():
        for x, y in loader:
            x, y = x.to(device), y.to(device)
            preds = model(x)
            loss = criterion(preds, y)
            total_loss += loss.item()
    return total_loss / len(loader)

In [ ]:
# Task 4: Define device, model, loss, optimizer:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = AgeRegressor(X_train.shape[1]).to(device)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

In [ ]:
# Run Training
num_epochs = 20
learning_rate = 0.001
# Define criterion (loss function)
criterion = nn.MSELoss()
# Define optimizer
optimizer = AdamW(model.parameters(), learning_rate)
train_losses = []
val_losses = []

print('Starting Training...')
for epoch in range(num_epochs):
  # Train one epoch
  train_loss = train_one_epoch(model, optimizer, criterion, train_loader, device)

  # Validate
  val_loss = test_ds(model, criterion, test_loader, device)

  train_losses.append(train_loss)
  val_losses.append(val_loss)

  if (epoch + 1) % 5 == 0:
    print(
      f'Epoch [{epoch+1}/{num_epochs}], '
      f'Train Loss: {train_loss:.4f}, '
      f'Val Loss: {val_loss:.4f}'
    )

print('Training Complete!')

In [ ]:
# Task 5: Start training for 20 epochs:
train_losses = []
val_losses = []

for epoch in range(20):
    train_loss = train_loop(model, train_loader, criterion, optimizer, device)
    val_loss = val_loop(model, test_loader, criterion, device)
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    print(f"Epoch {epoch+1}/20 | Train: {train_loss:.4f} | Val: {val_loss:.4f}")

In [ ]:
# Task 1: Write your code here:
plt.plot(train_losses, label="Train Loss")
plt.plot(val_losses, label="Validation Loss")
plt.xlabel("Epoch")
plt.ylabel("MSE Loss")
plt.legend()
plt.show()

In [ ]:
# Task 2 (Bonus): Write your code here: